
# Yambda 15 · SASRec candidates for the ranker-training window (Colab GPU)

Trains SASRec on `split_rank` (days < 259) with the validation settings (v1, 12 epochs) and exports
top-200 candidates. Needs in `MyDrive/music_recsys/`: `colab_bundle.zip`, the new `colab_src.zip`,
`colab_rank.zip`, and `outputs/best_sasrec.json`. Runtime → T4 GPU. ~10 min.


In [ ]:

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/music_recsys"
!rm -rf /content/music_recsys && mkdir -p /content/music_recsys
!unzip -q "{DRIVE}/colab_bundle.zip" -d /content/music_recsys
!unzip -q -o "{DRIVE}/colab_src.zip" -d /content/music_recsys
!unzip -q -o "{DRIVE}/colab_rank.zip" -d /content/music_recsys
!mkdir -p /content/music_recsys/reports/yambda && cp "{DRIVE}/outputs/best_sasrec.json" /content/music_recsys/reports/yambda/

import sys
sys.path.insert(0, "/content/music_recsys/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime!)")


In [ ]:

import json, time
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate
from musicrec.models.sasrec import SASRec
from musicrec.ranking import candidates_from_model

OUT = config.PROJECT_ROOT / "reports" / "yambda"
split = Split.load(config.YAMBDA_SPLIT_RANK_DIR)
v1 = json.loads((OUT / "best_sasrec.json").read_text())
sas = SASRec(**v1["params"], epochs=v1["best_epoch"], early_stopping=False)
t = time.time(); sas.fit(split); print(f"trained in {time.time() - t:.0f}s")
print("recall@10 on days 259-273:", round(evaluate(sas, split)["recall@10"], 4))
cands = candidates_from_model(sas, split, 200)
cands.to_parquet(OUT / "cands_rank_sasrec.parquet", index=False)
!mkdir -p "{DRIVE}/outputs_rank" && cp {OUT}/cands_rank_sasrec.parquet "{DRIVE}/outputs_rank/"
!ls -la "{DRIVE}/outputs_rank"
